In [3]:
import IPython
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import tir as T

变换批量矩阵乘法程序
现在，让我们回到 bmm_relu 练习。

首先，让我们看看 bmm 的定义:
- <math xmlns="http://www.w3.org/1998/Math/MathML">
  <msub>
    <mi>Y</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>i</mi>
      <mo>,</mo>
      <mi>j</mi>
    </mrow>
  </msub>
  <mo>=</mo>
  <munder>
    <mo data-mjx-texclass="OP">&#x2211;</mo>
    <mi>k</mi>
  </munder>
  <msub>
    <mi>A</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>i</mi>
      <mo>,</mo>
      <mi>k</mi>
    </mrow>
  </msub>
  <mo>&#xD7;</mo>
  <msub>
    <mi>B</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>k</mi>
      <mo>,</mo>
      <mi>j</mi>
    </mrow>
  </msub>
</math>

- <math xmlns="http://www.w3.org/1998/Math/MathML">
  <msub>
    <mi>C</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>i</mi>
      <mo>,</mo>
      <mi>j</mi>
    </mrow>
  </msub>
  <mo>=</mo>
  <mrow data-mjx-texclass="ORD">
    <mi mathvariant="double-struck">relu</mi>
  </mrow>
  <mo stretchy="false">(</mo>
  <msub>
    <mi>Y</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>i</mi>
      <mo>,</mo>
      <mi>j</mi>
    </mrow>
  </msub>
  <mo stretchy="false">)</mo>
  <mo>=</mo>
  <mrow data-mjx-texclass="ORD">
    <mi mathvariant="double-struck">max</mi>
  </mrow>
  <mo stretchy="false">(</mo>
  <msub>
    <mi>Y</mi>
    <mrow data-mjx-texclass="ORD">
      <mi>n</mi>
      <mo>,</mo>
      <mi>i</mi>
      <mo>,</mo>
      <mi>j</mi>
    </mrow>
  </msub>
  <mo>,</mo>
  <mn>0</mn>
  <mo stretchy="false">)</mo>
</math>

现在是你为 bmm_relu 编写 TensorIR 的时候了。我们提供 lnumpy 函数作为提示：

In [ ]:
def lnumpy_mm_relu_v2(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((16, 128, 128), dtype="float32")
    for n in range(16):
        for i in range(128):
            for j in range(128):
                for k in range(128):
                    if k == 0:
                        Y[n, i, j] = 0
                    Y[n, i, j] = Y[n, i, j] + A[n, i, k] * B[n, k, j]
                    
    for n in range(16):
        for i in range(128):
            for j in range(128):
                C[n, i, j] = max(Y[n, i, j], 0)

In [23]:
def lnumpy_mm_relu_v3(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    for n in range(16):
        for i in range(128):
            for j in range(128):
                for k in range(128):
                    if k == 0:
                        C[n, i, j] = 0
                    C[n, i, j] = C[n, i, j] + A[n, i, k] * B[n, k, j]
                C[n, i, j] = max(C[n, i, j], 0)

In [ ]:
@tvm.script.ir_module
class MyBmmRelu:
  @T.prim_func
  def bmm_relu( A:T.Buffer((16, 128, 128), "float32"),
                B:T.Buffer((16, 128, 128), "float32"),
                C:T.Buffer((16, 128, 128), "float32")):
    T.func_attr({"global_symbol": "bmm_relu", "tir.noalias": True})
    for n, i, j, k in T.grid(16, 128, 128, 128):
        with T.block("Y"):
            vn, vi, vj, vk = T.axis.remap("SSSR", [n, i, j, k])
            with T.init():
                C[vn, vi, vj] = T.float32(0)
            C[vn, vi, vj] = A[vn, vi, vk] * B[vn, vk, vj] + C[vn, vi, vj]
        with T.block("C"):
            vn, vi, vj = T.axis.remap("SSS", [n, i, j])
            C[vn, vi, vj] = T.max(C[vn, vi, vj], T.float32(0))


sch = tvm.tir.Schedule(MyBmmRelu)
IPython.display.Code(sch.mod.script(), language="python")
# Also please validate your result

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for n, i, j, k in T.grid(16, 128, 128, 128):
            with T.block("Y"):
                vn, vi, vj, vk = T.axis.remap("SSSR", [n, i, j, k])
                T.reads(A[vn, vi, vk], B[vn, vk, vj])
                T.writes(C[vn, vi, vj])
                with T.init():
                    C[vn, vi, vj] = T.float32(0.0)
                C[vn, vi, vj] = A[vn, vi, vk] * B[vn, vk, vj] + C[vn, vi, vj]
            with T.block("C"):
                vn, vi, vj = T.axis.remap("SSS", [n, i, j])
                T.reads(C[vn, vi, vj])
                T.writes(C[vn, vi, vj])
                C[vn, vi, vj] = T.max(C[vn, vi, vj], T.float32(0.0))

In [ ]:
sch = tvm.tir.Schedule(MyBmmRelu)
# TODO: transformations
# Hints: you can use
# `IPython.display.Code(sch.mod.script(), language="python")`
# or `print(sch.mod.script())`
# to show the current program at any time during the transformation.

# Step 1. Get blocks
Y = sch.get_block("Y", func_name="bmm_relu")

# Step 2. Get loops
n, i, j, k = sch.get_loops(Y)

# Step 3. Organize the loops
k0, k1 = sch.split(k, factors=[None, 4])
sch.reorder(...)
sch.compute_at/reverse_compute_at(...)
...

# Step 4. decompose reduction
Y_init = sch.decompose_reduction(Y, ...)
...

# Step 5. vectorize / parallel / unroll
sch.vectorize(...)
sch.parallel(...)
sch.unroll(...)
...

IPython.display.Code(sch.mod.script(), language="python")


In [ ]:
N, I, J = 16, 128 ,128
a_np = np.random.rand(16, 128, 128).astype("float32")
b_np = np.random.rand(16, 128, 128).astype("float32")
c_np = np.empty((N, I, J), dtype=np.float32)
lnumpy_mm_relu_v3(a_np, b_np, c_np)

before_rt_lib = tvm.build(MyBmmRelu, target="llvm")
after_rt_lib = tvm.build(sch.mod, target="llvm")
a_tvm = tvm.nd.array(a_np)
b_tvm = tvm.nd.array(b_np)
c_tvm = tvm.nd.array(c_np)



before_timer = before_rt_lib.time_evaluator("bmm_relu", tvm.cpu())
print("Before transformation:")
print(before_timer(a_tvm, b_tvm, c_tvm))
np.testing.assert_allclose(c_tvm, c_np, rtol=1e-5)


f_timer = after_rt_lib.time_evaluator("bmm_relu", tvm.cpu())
print("After transformation:")
print(f_timer(a_tvm, b_tvm, c_tvm))
np.testing.assert_allclose(c_tvm, c_np, rtol=1e-5)



Before transformation:
Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
  41.3608      41.3608      41.3608      41.3608       0.0000                  


有关第一版被注释掉的实现是因为把卷积核操作想成了Matmul 之后发现是×+的形式

这里可能产生的问题是 有关T.init()对C初始化的问题 如果di dj不设置为规约的话 会导致在di dj两层循环中C每次都被置零无法记录之前保存的和